# Mapping migration


## Set up

To get started on this notebook, you’ll need to restore any variables from previous notebooks to your workspace. To save time and memory, make sure to specify which variables you want to load.

In [1]:
%store -r

You will also need to import any libraries you are using in this notebook. In this notebook, you will need some additional packages to make interactive maps with vector data and access pre-defined month names.

In [2]:
### import packages

### libraries for dynamic mapping

### manage crs
import cartopy.crs as ccrs

### customize the dynamic plot
import panel as pn

### make dynamic plot with geodataframe
import hvplot.pandas
import holoviews as hv

### get month names
import calendar

### file management
import os
import pathlib

## Step 5: Make an interactive map of GBIF observations

### Step 5A: Create a simplified `GeoDataFrame` for plotting

Plotting larger files can be time consuming. The code below will streamline plotting with `hvplot` by simplifying the geometry, projecting it to a Mercator projection that is compatible with `geoviews`, and cropping off areas in the Arctic:

1. Simplify the ecoregions with `.simplify(.05)`, and save it back to the `geometry` column.
2. Change the Coordinate Reference System (CRS) to Mercator with `.to_crs(ccrs.Mercator())`.
3. Use the plotting code that is already in the cell to check that the plotting runs quickly (less than a minute) and looks the way you want, making sure to change `gdf` to your `GeoDataFrame` name.

In [3]:
### Had to import these again because map was outputting
### Had similar problem with file management packages in the previous notebook 
import hvplot.pandas
import holoviews as hv

### simplify the geometry to speed up processing
ecoreg_gdf.geometry = ecoreg_gdf.simplify(

    0.05, preserve_topology = False ### tells it not to preserve topology. You would want to preserve it for an analysis
)
### change the CRS to Mercator for mapping, Because we will put a basemap behind polygons
ecoreg_gdf = ecoreg_gdf.to_crs(ccrs.Mercator())

### check that the plot runs in a reasonable amount of time
ecoreg_gdf.hvplot(geo = True, crs = ccrs.Mercator()) ### geo = True so we can add a basemap

:Polygons   [Longitude,Latitude]

### Step 5B: Map migration over time

1. If applicable, replace any variable names with the names you defined previously.
2. Replace `column_name_used_for_ecoregion_color` and `column_name_used_for_slider` with the column names you wish to use.
3. Customize your plot with your choice of title, tile source, color map, and size.

Your plot will probably still change months very slowly in your Jupyter notebook, because it calculates each month’s plot as needed. Open up the saved HTML file to see faster performance!

In [4]:
### prep the data, then combine the occurrence data with the ecoregions

### rename columns in ecoregID to match occurrence_df
ecoreg_gdf = ecoreg_gdf.rename(columns = {'OBJECTID': 'ecoregion'})

### make ecoreg ID the index for merge with occurrence_df
ecoreg_gdf = ecoreg_gdf.set_index('ecoregion')

### join the occurrences with the plotting GeoDataFrame
### Don't need spatial join yet because occurrence_df is not a spatial data frame
occurrence_gdf = ecoreg_gdf.join(occurrence_df)


In [5]:
### activate panel and hvplot extensions you'll need for the interactive plot
pn.extension()
hv.extension("bokeh")

In [6]:
### plot the data

### get the plot bounds so they don't change with the slider
xmin, ymin, xmax, ymax = occurrence_gdf.total_bounds

### define the slider widget
slider = pn.widgets.DiscreteSlider(
    name = 'month', 
    options = {calendar.month_name[i]: i for i in range(1, 13)}
)

### plot occurrence by ecoregion and month
migration_plot = (
    occurrence_gdf
    .hvplot(

        ### Color by number of observations
        c = 'norm_occurrences',
            groupby = 'month',

        ### use background tiles, basemap from Open Street Map
        geo = True, crs = ccrs.Mercator(), tiles = 'OSM',

        ### add a title
        title = "Veery Migration Based on Annual Observations",

        ### set plot boundaries
        xlim = (xmin, xmax), ylim = (ymin, ymax),

        ### set dimensions of map
        frame_height = 600,
        frame_width = 900,

        ### put the widget at the bottom of the map
        widget_location = 'bottom',
        colorbar = False,
        widgets = {'column_name': slider}
    )
)

### wrap  plot in the Panel layout
interactive_migration_plot = pn.panel(migration_plot)

### check it out
interactive_migration_plot

BokehModel(combine_events=True, render_bundle={'docs_json': {'c7edb642-45dd-4e09-bcb4-5ee73a02202b': {'version…

Due to some annoying issues with permissions for the basemap, we can't export the map you just made as an .html file with the basemap. So we'll make another version, without a basemap, to export. We'll revisit this for your portfolio assignment to make a nice map with a basemap.

In [7]:
map_path = os.path.join(data_dir, "migration_plot.html")
map_path

'C:\\Users\\honey\\Earth_data\\veery_migration\\migration_plot.html'

In [8]:
migration_plot.save(
map_path,
embed = True,
resources = "inline",
)

In [ ]:
### make a path for your map to live
map_path = os.path.join(data_dir, "migration_plot.html")

### make a version for export, without the basemap
export_migration_plot = (
    occurrence_gdf
    .hvplot(     
    
    ### Color by number of observations
            c = 'norm_occurrences',
                groupby = 'month',
    
          ### add a title
            title = "Veery Migration Based on Annual Observations",
    
            ### set plot boundaries
            xlim = (xmin, xmax), ylim = (ymin, ymax),
    
            ### set dimensions of map
            frame_height = 600,
            frame_width = 900,
    
            ### put the widget at the bottom of the map
            widget_location = 'bottom',
            colorbar = False,
            widgets = {'column_name': slider}
        ))
    
### export the map as an .html file
export_migration_plot.save(
    map_path,
    embed = True,
    resources = "inline",
)

### Reflect and Respond #2: Interpret your map
What seasonal trends do you see in the GBIF observations? How does your map align with the broader literature on Veery migration?